In [1]:
# Instalacao opcional
RUN_OPTIONAL_INSTALLS = False

if RUN_OPTIONAL_INSTALLS:
    %pip install -q -U pyarrow librosa tqdm
    print('[OK] Pacotes instalados.')
else:
    print('[OK] Instalacao ignorada. Ative RUN_OPTIONAL_INSTALLS=True se necessario.')


[OK] Instalacao ignorada. Ative RUN_OPTIONAL_INSTALLS=True se necessario.


# Fingerprint Band Rank — Geracao de `fingerprint_band_rank.parquet`

Pipeline que segmenta cada musica em blocos de 10 segundos (inicio em 15s),
calcula a STFT localmente em cada bloco com `center=False`, extrai os principais
picos por banda de frequencia e alinha com as anotacoes temporais de valence e
arousal do DEAM.

**Saidas:**
- `fingerprint_band_rank/fingerprint_band_rank.parquet` — consolidado com todos os blocos.
- `fingerprint_band_rank/songs/song_XXX_fingerprint_band_rank.parquet` — um arquivo por musica.


## Metodologia

No pipeline de fingerprint por ranking em bandas, cada musica foi segmentada em blocos
consecutivos de 10 segundos, sem sobreposicao entre blocos, iniciando em 15 segundos
para manter o alinhamento com as anotacoes temporais de valence e arousal do conjunto DEAM.

Para evitar contaminacao entre segmentos adjacentes, a STFT foi recalculada individualmente
em cada bloco, utilizando `center=False`. Como as anotacoes emocionais podem variar ao longo
da musica, cada bloco foi tratado como uma unidade temporal independente.

Entretanto, para preservar a dinamica relativa entre blocos da mesma musica, as magnitudes
em dB foram calculadas usando uma referencia global por musica. Em paralelo, foram mantidas
medidas normalizadas localmente por bloco, representando a saliencia relativa das frequencias
e bandas dentro de cada segmento.

Em seguida, o espectro foi dividido em bandas de frequencia, e os principais picos foram
extraidos separadamente em cada banda. Para cada bloco, os valores de valence e arousal foram
calculados pela media das anotacoes temporais correspondentes ao mesmo intervalo.

Como resultado, foi gerada uma unica base em formato Parquet chamada
`fingerprint_band_rank.parquet`, com granularidade por bloco, contendo metadados temporais,
rotulos emocionais e features de fingerprint organizadas por bandas de frequencia.


## Representacoes geradas pelo notebook

Este notebook gera duas representacoes complementares do fingerprint por banda.

**fingerprint_band_rank.parquet**: resumo por bloco com estatisticas por banda.

**fingerprint_band_rank_raw_peaks.parquet**: uma linha por pico por banda.
O pico e identificado no frame de maior amplitude (argmax) dentro do bloco.

### Regras metodologicas

- Blocos sem sobreposicao temporal (BLOCK_STEP_SEC == BLOCK_SIZE_SEC).
- STFT com sobreposicao interna entre frames (hop_length < n_fft) — normal e necessario.
- TOPK_PER_BAND nao foi reduzido.

### Aviso de vazamento de dados

- valence, arousal e quadrant_label mantidos apenas para rastreabilidade.
- Devem ser removidos das features antes de qualquer treinamento supervisionado.
- O split treino/teste deve ser feito por song_id, nunca por linha individual.

In [2]:
import os
import re
import math
import warnings
import pathlib

import numpy as np
import pandas as pd

try:
    import librosa
except ImportError:
    librosa = None
    print('[WARN] librosa nao instalado. Ative RUN_OPTIONAL_INSTALLS=True.')

warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

print('Importacoes carregadas')


Importacoes carregadas


In [3]:
# Execucao local: nenhum Google Drive precisa ser montado.
print('Execucao local configurada; Google Drive nao e necessario.')


Execucao local configurada; Google Drive nao e necessario.


In [4]:
# ============================================================
# CONFIGURACOES GLOBAIS
# ============================================================

RANDOM_STATE = 42

# Segmentacao de blocos
AUDIO_START_SEC       = 15.0   # bloco 0: 15s -> 25s, bloco 1: 25s -> 35s, ...
BLOCK_SIZE_SEC        = 10.0
BLOCK_STEP_SEC        = 10.0   # sem sobreposicao
MIN_BLOCK_DURATION_SEC = 5.0   # blocos finais menores que isso sao descartados

# STFT
N_FFT_GLOBAL     = 2048
HOP_LENGTH_GLOBAL = 1024
WINDOW_TYPE      = 'hann'
SR_TARGET        = 44100

EPS     = 1e-10
NYQUIST = SR_TARGET / 2.0

# Numero de picos por banda
TOPK_PER_BAND = 10

# Bandas de frequencia (Hz):
#   low       :   20 -  250 Hz
#   mid       :  250 - 2000 Hz
#   high      : 2000 - 8000 Hz
#   very_high : 8000 - Nyquist
FREQ_BANDS = {
    'low'      : (20.0,    250.0),
    'mid'      : (250.0,  2000.0),
    'high'     : (2000.0, 8000.0),
    'very_high': (8000.0, NYQUIST),
}

FINGERPRINT_METHOD = 'fingerprint_band_rank'

# 0 = processar tudo; use numero pequeno para teste rapido
MAX_FILES_TO_PROCESS = 0


# Diretorio raiz local do dataset
BASE_DIR = pathlib.Path(r"C:\dev\python\TCC Ajustado\Dados")

FINGERPRINT_DIR = BASE_DIR / 'fingerprint_band_rank'
FINGERPRINT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET_NAME = 'fingerprint_band_rank.parquet'

print('[OK] Configuracoes carregadas')
print(f'     BASE_DIR              = {BASE_DIR}')
print(f'     FINGERPRINT_DIR       = {FINGERPRINT_DIR}')
print(f'     AUDIO_START_SEC       = {AUDIO_START_SEC}s')
print(f'     BLOCK_SIZE_SEC        = {BLOCK_SIZE_SEC}s')
print(f'     BLOCK_STEP_SEC        = {BLOCK_STEP_SEC}s')
print(f'     MIN_BLOCK_DURATION    = {MIN_BLOCK_DURATION_SEC}s')
print(f'     TOPK_PER_BAND         = {TOPK_PER_BAND}')
print(f'     N_FFT={N_FFT_GLOBAL} | HOP_LENGTH={HOP_LENGTH_GLOBAL} | WINDOW={WINDOW_TYPE}')


[OK] Configuracoes carregadas
     BASE_DIR              = C:\dev\python\TCC Ajustado\Dados
     FINGERPRINT_DIR       = C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank
     AUDIO_START_SEC       = 15.0s
     BLOCK_SIZE_SEC        = 10.0s
     BLOCK_STEP_SEC        = 10.0s
     MIN_BLOCK_DURATION    = 5.0s
     TOPK_PER_BAND         = 10
     N_FFT=2048 | HOP_LENGTH=1024 | WINDOW=hann


In [5]:
# ============================================================
# CAMINHOS LOCAIS DE ENTRADA E CARREGAMENTO DEAM
# ============================================================

IN_PATH   = str(BASE_DIR / 'audio')
ANNOT_DIR = BASE_DIR
V_CSV    = BASE_DIR / 'valence.csv'
A_CSV    = BASE_DIR / 'arousal.csv'
OUT_FILE = FINGERPRINT_DIR / OUTPUT_PARQUET_NAME

print(f'Audio  : {IN_PATH}')
print(f'Valence: {V_CSV}')
print(f'Arousal: {A_CSV}')
print(f'Saida  : {OUT_FILE}')

ok_audio = os.path.isdir(IN_PATH)
ok_v     = os.path.exists(V_CSV)
ok_a     = os.path.exists(A_CSV)
print(f'\nDiretorio de audio existe : {ok_audio}')
print(f'valence.csv existe         : {ok_v}')
print(f'arousal.csv existe         : {ok_a}')


def _melt_deam(df, col_name):
    m = df.melt(id_vars='song_id', var_name='ts', value_name=col_name)
    m['time_s'] = m['ts'].str.extract(r'(\d+)').astype(float) / 1000.0
    return m.drop(columns='ts')


DEAM_VA = None
if ok_v and ok_a:
    try:
        df_v_raw = pd.read_csv(V_CSV)
        df_a_raw = pd.read_csv(A_CSV)
        DEAM_VA = pd.merge(
            _melt_deam(df_v_raw, 'valence'),
            _melt_deam(df_a_raw, 'arousal'),
            on=['song_id', 'time_s'],
            how='inner',
        )
        print(f'\n[OK] Anotacoes V/A: {len(DEAM_VA)} pontos | {DEAM_VA["song_id"].nunique()} musicas')
    except Exception as e:
        print(f'[ERR] Falha ao carregar V/A: {e}')
else:
    print('[WARN] valence.csv ou arousal.csv nao encontrados — DEAM_VA = None')


Audio  : C:\dev\python\TCC Ajustado\Dados\audio
Valence: C:\dev\python\TCC Ajustado\Dados\valence.csv
Arousal: C:\dev\python\TCC Ajustado\Dados\arousal.csv
Saida  : C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\fingerprint_band_rank.parquet

Diretorio de audio existe : True
valence.csv existe         : True
arousal.csv existe         : True

[OK] Anotacoes V/A: 2203846 pontos | 1802 musicas


In [6]:
# ============================================================
# FUNCOES UTILITARIAS
# ============================================================


def _iter_audio_files(in_path):
    p = pathlib.Path(in_path)
    files = []
    for ext in ['*.mp3', '*.wav', '*.ogg', '*.flac']:
        files.extend(list(p.glob(ext)))
    files = sorted(files)
    result = []
    for fp in files:
        m = re.search(r'(\d+)', fp.name)
        sid = int(m.group(1)) if m else fp.stem
        result.append((str(fp), sid))
    return result


def freq_to_midi(freq_hz):
    try:
        f = float(freq_hz)
        if not math.isfinite(f) or f <= 0:
            return float('nan')
        return 69.0 + 12.0 * math.log2(f / 440.0)
    except Exception:
        return float('nan')


def midi_to_octave(midi):
    try:
        m = float(midi)
        if not math.isfinite(m):
            return float('nan')
        return float(math.floor(m / 12.0) - 1.0)
    except Exception:
        return float('nan')


def midi_to_semitone(midi):
    try:
        m = float(midi)
        if not math.isfinite(m):
            return float('nan')
        return float(round(m))
    except Exception:
        return float('nan')


def midi_to_pitch_class(midi):
    try:
        m = float(midi)
        if not math.isfinite(m):
            return None
        return int(round(m)) % 12
    except Exception:
        return None


def compute_quadrant(valence, arousal):
    """Classifica quadrante emocional segundo o modelo circumplex de Russell."""
    if pd.isna(valence) or pd.isna(arousal):
        return 'INDEFINIDO', 'INDEFINIDO'
    v, a = float(valence), float(arousal)
    if v >= 0 and a >= 0:
        return 'Q1', 'Q1_High_Valence_High_Arousal'
    elif v < 0 and a >= 0:
        return 'Q2', 'Q2_Low_Valence_High_Arousal'
    elif v < 0 and a < 0:
        return 'Q3', 'Q3_Low_Valence_Low_Arousal'
    else:
        return 'Q4', 'Q4_High_Valence_Low_Arousal'


def get_band_indices(freqs, f_low, f_high):
    """Retorna indices de freqs na banda [f_low, f_high)."""
    f_high_eff = f_high if (f_high is not None and math.isfinite(f_high)) else float('inf')
    return np.where((freqs >= f_low) & (freqs < f_high_eff))[0]


print('Funcoes utilitarias carregadas')


Funcoes utilitarias carregadas


In [7]:
# ============================================================
# FUNCOES DE PERSISTENCIA POR MUSICA
# Estrutura: FINGERPRINT_DIR / songs / song_XXX_fingerprint_band_rank.parquet
# ============================================================


def _song_tag(song_id):
    try:
        return f'song_{int(song_id):03d}'
    except Exception:
        return f'song_{str(song_id)}'


def _song_dir():
    """Pasta compartilhada songs/ dentro de fingerprint_band_rank."""
    d = FINGERPRINT_DIR / 'songs'
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_song_band_rank_parquet(song_rows, song_id):
    """Salva todos os blocos de um song_id em Parquet proprio dentro de songs/."""
    if not song_rows:
        return None
    df_song = pd.DataFrame(song_rows)
    df_song['song_id']  = df_song['song_id'].astype(int)
    df_song['block_id'] = df_song['block_id'].astype(int)
    df_song = df_song.sort_values(['song_id', 'block_id']).reset_index(drop=True)
    tag = _song_tag(song_id)
    song_path = _song_dir() / f'{tag}_fingerprint_band_rank.parquet'
    df_song.to_parquet(song_path, index=False, engine='pyarrow', compression='snappy')
    return song_path


print('Funcoes de persistencia por musica carregadas')
print(f'     Estrutura: {FINGERPRINT_DIR}/songs/song_XXX_fingerprint_band_rank.parquet')


Funcoes de persistencia por musica carregadas
     Estrutura: C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank/songs/song_XXX_fingerprint_band_rank.parquet


In [8]:
# ============================================================
# FUNCAO DE PROCESSAMENTO DE BLOCO
# Retorna (row, df_raw_peaks_block) em vez de apenas row.
# ============================================================

# Mapeamento de banda para ID inteiro (deterministico)
_BAND_ID_MAP_BAND = {name: i for i, name in enumerate(FREQ_BANDS.keys())}


def process_block(
    y_block, ref_song, freqs, song_id, filename,
    block_id, block_start_sec, block_end_sec,
    valence, arousal,
):
    # Processa um bloco de audio e retorna (row, df_raw_peaks_block).
    #
    # row              : dict com features resumidas por bloco.
    # df_raw_peaks_block: DataFrame com uma linha por pico por banda (raw_peaks).
    #
    # Para cada top-K bin de frequencia em cada banda, o pico e identificado no
    # frame de maior amplitude (argmax) dentro do bloco. Isso garante um ponto
    # tempo-frequencia real, nao apenas uma media ao longo do bloco.
    #
    # magnitude      : S_abs no ponto (freq_bin, frame_max) — amplitude bruta.
    # magnitude_db   : S_db  no ponto (freq_bin, frame_max) — dB relativo a ref global.
    # magnitude_norm : S_norm no ponto (freq_bin, frame_max) — normalizado pelo bloco [0,1].
    #
    # AVISO DE VAZAMENTO DE DADOS:
    #   valence, arousal, quadrant_label sao mantidos apenas para rastreabilidade.
    #   Devem ser REMOVIDOS antes de qualquer treinamento supervisionado.
    #   Split deve ser feito por song_id, NUNCA por linha.
    block_duration_sec = block_end_sec - block_start_sec
    quadrant, quadrant_label = compute_quadrant(valence, arousal)

    # 1. STFT local no bloco — evita contaminacao entre blocos adjacentes
    S_complex = librosa.stft(
        y_block,
        n_fft=N_FFT_GLOBAL,
        hop_length=HOP_LENGTH_GLOBAL,
        window=WINDOW_TYPE,
        center=False,
    )
    S_abs = np.abs(S_complex).astype(np.float32)
    del S_complex

    # 2. dB com referencia global da musica (preserva intensidade relativa entre blocos)
    S_db = librosa.amplitude_to_db(S_abs, ref=ref_song)

    # 3. Normalizacao local: escala [0, 1] pelo maximo do proprio bloco
    ref_block = max(float(np.max(S_abs)), EPS)
    S_norm = (S_abs / ref_block).astype(np.float32)

    # 4. Timestamps locais dos frames (relativos ao inicio do bloco, em segundos)
    times_local = librosa.frames_to_time(
        np.arange(S_abs.shape[1]), sr=SR_TARGET, hop_length=HOP_LENGTH_GLOBAL
    ).astype(np.float32)

    row = {
        'song_id'           : int(song_id),
        'filename'          : filename,
        'block_id'          : int(block_id),
        'block_start_sec'   : float(block_start_sec),
        'block_end_sec'     : float(block_end_sec),
        'block_duration_sec': float(block_duration_sec),
        'valence'           : float(valence),
        'arousal'           : float(arousal),
        'quadrant'          : quadrant,
        'quadrant_label'    : quadrant_label,
        'method'            : FINGERPRINT_METHOD,
    }

    # 5. Energia por banda
    band_norm_energies = {}
    for band_name, (f_low, f_high) in FREQ_BANDS.items():
        idx = get_band_indices(freqs, f_low, f_high)
        if idx.size == 0:
            row[f'energy_db_{band_name}']   = float('nan')
            row[f'energy_norm_{band_name}'] = float('nan')
            band_norm_energies[band_name] = 0.0
        else:
            row[f'energy_db_{band_name}']   = float(np.nanmean(S_db[idx, :]))
            en = float(np.nanmean(S_norm[idx, :]))
            row[f'energy_norm_{band_name}'] = en
            band_norm_energies[band_name] = en

    # 6. Score por banda
    total_norm = sum(band_norm_energies.values()) + EPS
    for band_name in FREQ_BANDS:
        row[f'score_{band_name}'] = band_norm_energies[band_name] / total_norm

    # 7. Top-K picos por banda e estatisticas
    all_freq, all_mag_db, all_mag_norm = [], [], []
    raw_peaks_list = []   # acumula picos brutos para df_raw_peaks_block

    for band_name, (f_low, f_high) in FREQ_BANDS.items():
        idx = get_band_indices(freqs, f_low, f_high)

        if idx.size == 0:
            for k in range(1, TOPK_PER_BAND + 1):
                pfx = f'fp_{band_name}_top_{k}'
                for col in ('freq_hz', 'magnitude_db', 'magnitude_norm_block',
                             'rank', 'midi', 'octave', 'semitone_approx'):
                    row[f'{pfx}_{col}'] = float('nan')
            for stat in ('freq_mean', 'freq_std', 'mag_mean_db', 'mag_std_db',
                          'mag_mean_norm_block', 'mag_std_norm_block',
                          'octave_mean', 'octave_std', 'midi_mean', 'midi_std'):
                row[f'fp_{band_name}_{stat}'] = float('nan')
            row[f'fp_{band_name}_event_count'] = 0
            row[f'fp_{band_name}_unique_pitch_classes'] = 0
            continue

        # Media da amplitude absoluta por bin de frequencia ao longo dos frames do bloco
        mean_abs_per_bin = S_abs[idx, :].mean(axis=1)

        top_k_n    = min(TOPK_PER_BAND, len(idx))
        top_local  = np.argsort(mean_abs_per_bin)[::-1][:top_k_n]
        top_global = idx[top_local]

        freq_list, mag_db_list, mag_norm_list = [], [], []

        for rank_k, global_i in enumerate(top_global, start=1):
            f_hz       = float(freqs[global_i])
            mag_db_v   = float(np.nanmean(S_db[global_i, :]))
            mag_norm_v = float(np.nanmean(S_norm[global_i, :]))
            midi_v     = freq_to_midi(f_hz)
            octave_v   = midi_to_octave(midi_v)
            semi_v     = midi_to_semitone(midi_v)

            pfx = f'fp_{band_name}_top_{rank_k}'
            row[f'{pfx}_freq_hz']              = f_hz
            row[f'{pfx}_magnitude_db']         = mag_db_v
            row[f'{pfx}_magnitude_norm_block'] = mag_norm_v
            row[f'{pfx}_rank']                 = float(rank_k)
            row[f'{pfx}_midi']                 = midi_v   if math.isfinite(midi_v)   else float('nan')
            row[f'{pfx}_octave']               = octave_v if math.isfinite(octave_v) else float('nan')
            row[f'{pfx}_semitone_approx']      = semi_v   if math.isfinite(semi_v)   else float('nan')

            freq_list.append(f_hz)
            mag_db_list.append(mag_db_v)
            mag_norm_list.append(mag_norm_v)

            # --- Pico bruto: frame de maximo para este bin de frequencia ---
            # Garante um ponto tempo-frequencia real (nao uma media temporal).
            if times_local.size > 0:
                frame_max = int(np.argmax(S_abs[global_i, :]))
                raw_peaks_list.append({
                    'band_name'              : band_name,
                    'band_id'                : int(_BAND_ID_MAP_BAND.get(band_name, -1)),
                    'band_low_hz'            : float(f_low),
                    'band_high_hz'           : float(f_high),
                    'peak_rank_in_band'      : int(rank_k),
                    'topk_per_band'          : int(TOPK_PER_BAND),
                    'frame_idx'              : int(frame_max),
                    'freq_bin'               : int(global_i),
                    'frequency_hz'           : float(f_hz),
                    'magnitude'              : float(S_abs[global_i, frame_max]),
                    'magnitude_db'           : float(S_db[global_i, frame_max]),
                    'magnitude_norm'         : float(S_norm[global_i, frame_max]),
                    'peak_time_relative_sec' : float(times_local[frame_max]),
                    'peak_time_sec'          : float(block_start_sec + times_local[frame_max]),
                })

        # Preenche com NaN as posicoes alem dos bins disponiveis na banda
        for rank_k in range(top_k_n + 1, TOPK_PER_BAND + 1):
            pfx = f'fp_{band_name}_top_{rank_k}'
            for col in ('freq_hz', 'magnitude_db', 'magnitude_norm_block',
                         'rank', 'midi', 'octave', 'semitone_approx'):
                row[f'{pfx}_{col}'] = float('nan')

        # Estatisticas da banda (sobre os top-K bins)
        fa  = np.array(freq_list, dtype=float)
        da  = np.array(mag_db_list, dtype=float)
        na  = np.array(mag_norm_list, dtype=float)
        ma  = np.array([freq_to_midi(f) for f in freq_list], dtype=float)
        oa  = np.array([midi_to_octave(m) for m in ma], dtype=float)
        pca = [midi_to_pitch_class(m) for m in ma]

        row[f'fp_{band_name}_freq_mean']           = float(np.nanmean(fa))
        row[f'fp_{band_name}_freq_std']            = float(np.nanstd(fa))
        row[f'fp_{band_name}_mag_mean_db']         = float(np.nanmean(da))
        row[f'fp_{band_name}_mag_std_db']          = float(np.nanstd(da))
        row[f'fp_{band_name}_mag_mean_norm_block'] = float(np.nanmean(na))
        row[f'fp_{band_name}_mag_std_norm_block']  = float(np.nanstd(na))
        row[f'fp_{band_name}_event_count']         = len(freq_list)
        row[f'fp_{band_name}_octave_mean']         = float(np.nanmean(oa))
        row[f'fp_{band_name}_octave_std']          = float(np.nanstd(oa))
        row[f'fp_{band_name}_midi_mean']           = float(np.nanmean(ma))
        row[f'fp_{band_name}_midi_std']            = float(np.nanstd(ma))
        row[f'fp_{band_name}_unique_pitch_classes'] = int(
            len(set(pc for pc in pca if pc is not None))
        )

        all_freq.extend(freq_list)
        all_mag_db.extend(mag_db_list)
        all_mag_norm.extend(mag_norm_list)

    # 8. Estatisticas globais do bloco (todas as bandas)
    if all_freq:
        row['fp_total_event_count']          = len(all_freq)
        row['fp_total_mag_mean_db']          = float(np.nanmean(all_mag_db))
        row['fp_total_mag_std_db']           = float(np.nanstd(all_mag_db))
        row['fp_total_mag_mean_norm_block']  = float(np.nanmean(all_mag_norm))
        row['fp_total_mag_std_norm_block']   = float(np.nanstd(all_mag_norm))
        row['fp_total_freq_mean']            = float(np.nanmean(all_freq))
        row['fp_total_freq_std']             = float(np.nanstd(all_freq))
    else:
        row['fp_total_event_count'] = 0
        for col in ('fp_total_mag_mean_db', 'fp_total_mag_std_db',
                    'fp_total_mag_mean_norm_block', 'fp_total_mag_std_norm_block',
                    'fp_total_freq_mean', 'fp_total_freq_std'):
            row[col] = float('nan')

    # 9. Construir DataFrame de picos brutos para este bloco
    if raw_peaks_list:
        df_raw = pd.DataFrame(raw_peaks_list)
        # Colunas de identificacao e contexto
        df_raw.insert(0, 'song_id',         int(song_id))
        df_raw.insert(1, 'filename',        filename)
        df_raw.insert(2, 'block_id',        int(block_id))
        df_raw.insert(3, 'block_start_sec', float(block_start_sec))
        df_raw.insert(4, 'block_end_sec',   float(block_end_sec))
        # Informacao musical
        df_raw['midi_note']   = df_raw['frequency_hz'].apply(freq_to_midi).astype(np.float32)
        df_raw['octave']      = df_raw['midi_note'].apply(midi_to_octave).astype(np.float32)
        df_raw['semitone']    = df_raw['midi_note'].apply(midi_to_semitone).astype(np.float32)
        df_raw['pitch_class'] = df_raw['midi_note'].apply(midi_to_pitch_class).astype(np.float32)
        # Ranking global entre todos os picos do bloco
        df_raw['peak_rank_global'] = (
            df_raw['magnitude_db']
            .rank(ascending=False, method='first')
            .astype(np.int32)
        )
        # Labels emocionais — apenas para rastreabilidade; remover antes de treinar
        val_f = float(valence) if pd.notna(valence) else float('nan')
        aro_f = float(arousal) if pd.notna(arousal) else float('nan')
        df_raw['valence']        = np.float32(val_f)
        df_raw['arousal']        = np.float32(aro_f)
        df_raw['quadrant_label'] = quadrant_label
        # Otimizacao de tipos
        for col in ('band_name', 'quadrant_label', 'filename'):
            if col in df_raw.columns:
                df_raw[col] = df_raw[col].astype('category')
        for col in ('magnitude', 'magnitude_db', 'magnitude_norm',
                    'frequency_hz', 'peak_time_sec', 'peak_time_relative_sec',
                    'band_low_hz', 'band_high_hz'):
            if col in df_raw.columns:
                df_raw[col] = df_raw[col].astype(np.float32)
        df_raw_peaks_block = df_raw
    else:
        df_raw_peaks_block = pd.DataFrame()

    return row, df_raw_peaks_block


print('Funcao process_block carregada (retorna row, df_raw_peaks_block)')


Funcao process_block carregada (retorna row, df_raw_peaks_block)


In [9]:
# ============================================================
# PIPELINE PRINCIPAL — LOOP POR MUSICA
# ============================================================

# Garantia metodologica: blocos do raw_peaks nao podem se sobrepor
assert BLOCK_STEP_SEC == BLOCK_SIZE_SEC, (
    "raw_peaks requer blocos sem sobreposicao temporal. "
    "Use BLOCK_STEP_SEC igual a BLOCK_SIZE_SEC."
)

if librosa is None:
    raise ImportError('librosa nao instalado. Ative RUN_OPTIONAL_INSTALLS=True.')

all_rows = []
raw_band_peaks_dfs = []   # acumula picos brutos de cada bloco

stats = {
    'songs_processed'       : 0,
    'songs_skipped_short'   : 0,
    'blocks_total'          : 0,
    'blocks_discarded_short': 0,
    'blocks_discarded_no_va': 0,
}

audio_files = _iter_audio_files(IN_PATH)
if MAX_FILES_TO_PROCESS > 0:
    audio_files = audio_files[:MAX_FILES_TO_PROCESS]

print(f'Arquivos encontrados : {len(audio_files)}')
print(f'AUDIO_START_SEC      = {AUDIO_START_SEC}s')
print(f'BLOCK_SIZE_SEC       = {BLOCK_SIZE_SEC}s  (passo={BLOCK_STEP_SEC}s, min={MIN_BLOCK_DURATION_SEC}s)')
print()

_iter = tqdm(audio_files, desc='Processando musicas') if tqdm is not None else audio_files

for audio_path, song_id in _iter:
    filename = os.path.basename(audio_path)
    song_rows = []

    try:
        y_full, sr = librosa.load(audio_path, sr=SR_TARGET, mono=True)
    except Exception as e:
        print(f'[ERR] {filename}: {e}')
        continue

    duration = float(len(y_full)) / sr
    if duration <= AUDIO_START_SEC:
        if tqdm is None:
            print(f'[SKIP] {filename}: duracao {duration:.1f}s <= AUDIO_START_SEC')
        stats['songs_skipped_short'] += 1
        continue

    try:
        S_complex_song = librosa.stft(
            y_full,
            n_fft=N_FFT_GLOBAL,
            hop_length=HOP_LENGTH_GLOBAL,
            window=WINDOW_TYPE,
            center=False,
        )
        S_abs_song = np.abs(S_complex_song)
        ref_song = max(float(np.max(S_abs_song)), EPS)
        freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT_GLOBAL)
        del S_complex_song, S_abs_song
    except Exception as e:
        print(f'[ERR] STFT global {filename}: {e}')
        continue

    va_song = None
    if DEAM_VA is not None:
        va_tmp = DEAM_VA[DEAM_VA['song_id'] == song_id]
        if not va_tmp.empty:
            va_song = va_tmp.reset_index(drop=True)

    start_sample  = int(AUDIO_START_SEC * sr)
    block_samples = int(BLOCK_SIZE_SEC * sr)
    step_samples  = int(BLOCK_STEP_SEC * sr)

    block_id = 0
    song_blocks      = 0
    song_disc_short  = 0
    song_disc_no_va  = 0

    block_start_sample = start_sample
    while block_start_sample < len(y_full):
        block_end_sample = block_start_sample + block_samples
        y_block = y_full[block_start_sample : min(block_end_sample, len(y_full))]

        block_start_sec = block_start_sample / sr
        block_end_sec   = min(block_end_sample, len(y_full)) / sr
        block_duration  = block_end_sec - block_start_sec

        if block_duration < MIN_BLOCK_DURATION_SEC:
            song_disc_short += 1
            stats['blocks_discarded_short'] += 1
            break

        valence_blk = float('nan')
        arousal_blk = float('nan')
        if va_song is not None:
            va_win = va_song[
                (va_song['time_s'] >= block_start_sec) &
                (va_song['time_s'] <  block_end_sec)
            ]
            if not va_win.empty:
                valence_blk = float(va_win['valence'].mean())
                arousal_blk = float(va_win['arousal'].mean())

        if pd.isna(valence_blk) or pd.isna(arousal_blk):
            song_disc_no_va += 1
            stats['blocks_discarded_no_va'] += 1
            block_id += 1
            block_start_sample += step_samples
            continue

        # Processa bloco — retorna (row, df_raw_peaks_block)
        try:
            row, df_raw_block = process_block(
                y_block         = y_block,
                ref_song        = ref_song,
                freqs           = freqs,
                song_id         = song_id,
                filename        = filename,
                block_id        = block_id,
                block_start_sec = block_start_sec,
                block_end_sec   = block_end_sec,
                valence         = valence_blk,
                arousal         = arousal_blk,
            )
            all_rows.append(row)
            song_rows.append(row)
            song_blocks += 1
            stats['blocks_total'] += 1

            # Acumula picos brutos do bloco
            if df_raw_block is not None and not df_raw_block.empty:
                raw_band_peaks_dfs.append(df_raw_block)
        except Exception as e:
            print(f'[WARN] {filename} bloco {block_id}: {e}')

        block_id += 1
        block_start_sample += step_samples

    song_path = save_song_band_rank_parquet(song_rows, song_id)
    if song_path is not None:
        print(f'[OK] song {song_id} | blocos={song_blocks} | desc_dur={song_disc_short} | desc_va={song_disc_no_va} | {song_path.absolute()}')
    else:
        print(f'[WARN] song {song_id}: nenhum bloco valido')

    stats['songs_processed'] += 1

print()
print(f'[OK] Pipeline concluido!')
print(f'     Musicas processadas      : {stats["songs_processed"]}')
print(f'     Blocos gerados           : {stats["blocks_total"]}')
print(f'     Desc. por duracao curta  : {stats["blocks_discarded_short"]}')
print(f'     Desc. por ausencia V/A   : {stats["blocks_discarded_no_va"]}')
print(f'     Total de picos brutos    : {sum(len(d) for d in raw_band_peaks_dfs)}')


Arquivos encontrados : 1802
AUDIO_START_SEC      = 15.0s
BLOCK_SIZE_SEC       = 10.0s  (passo=10.0s, min=5.0s)



Processando musicas:   0%|          | 0/1802 [00:00<?, ?it/s]

[OK] song 10 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs\song_010_fingerprint_band_rank.parquet
[OK] song 1000 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs\song_1000_fingerprint_band_rank.parquet
[OK] song 1001 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs\song_1001_fingerprint_band_rank.parquet
[OK] song 1002 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs\song_1002_fingerprint_band_rank.parquet
[OK] song 1003 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs\song_1003_fingerprint_band_rank.parquet
[OK] song 1004 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs\song_1004_fingerprint_band_rank.parquet
[OK] song 1005 | blocos=3 | desc_dur=1 | desc_va=0 | C:\dev\python\TCC Ajustado

In [10]:
# ============================================================
# SALVAR fingerprint_band_rank.parquet
# ============================================================

if not all_rows:
    print('[WARN] Nenhum bloco processado. Verifique os caminhos de entrada.')
    df_band_rank = pd.DataFrame()
else:
    df_band_rank = pd.DataFrame(all_rows)
    df_band_rank['song_id']  = df_band_rank['song_id'].astype(int)
    df_band_rank['block_id'] = df_band_rank['block_id'].astype(int)
    df_band_rank = df_band_rank.sort_values(['song_id', 'block_id']).reset_index(drop=True)

    out_path = FINGERPRINT_DIR / OUTPUT_PARQUET_NAME
    df_band_rank.to_parquet(out_path, index=False, engine='pyarrow', compression='snappy')

    print(f'Salvo: {out_path}')
    print(f'Shape: {df_band_rank.shape}')
    print(f'Musicas  : {df_band_rank["song_id"].nunique()}')
    print(f'Blocos   : {len(df_band_rank)}')
    print(f'Colunas  : {len(df_band_rank.columns)}')


Salvo: C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\fingerprint_band_rank.parquet
Shape: (6506, 358)
Musicas  : 1802
Blocos   : 6506
Colunas  : 358


In [11]:
# ============================================================
# SALVAR fingerprint_band_rank_raw_peaks.parquet
# ============================================================

RAW_BAND_PEAKS_PARQUET_NAME = 'fingerprint_band_rank_raw_peaks.parquet'
raw_band_peaks_out_path = FINGERPRINT_DIR / RAW_BAND_PEAKS_PARQUET_NAME

if not raw_band_peaks_dfs:
    print('[WARN] Nenhum pico bruto acumulado. fingerprint_band_rank_raw_peaks.parquet nao sera gerado.')
    df_band_raw_peaks = pd.DataFrame()
else:
    df_band_raw_peaks = pd.concat(raw_band_peaks_dfs, ignore_index=True)
    df_band_raw_peaks['song_id']  = df_band_raw_peaks['song_id'].astype(int)
    df_band_raw_peaks['block_id'] = df_band_raw_peaks['block_id'].astype(np.int32)
    df_band_raw_peaks = df_band_raw_peaks.sort_values(
        ['song_id', 'block_id', 'band_id', 'peak_rank_in_band']
    ).reset_index(drop=True)

    df_band_raw_peaks.to_parquet(
        raw_band_peaks_out_path,
        index=False,
        engine='pyarrow',
        compression='snappy',
    )

    print(f'[OK] {RAW_BAND_PEAKS_PARQUET_NAME} salvo!')
    print(f'     Shape   : {df_band_raw_peaks.shape[0]} picos x {df_band_raw_peaks.shape[1]} colunas')
    print(f'     Songs   : {df_band_raw_peaks["song_id"].nunique()}')
    print(f'     Blocos  : {df_band_raw_peaks[["song_id","block_id"]].drop_duplicates().shape[0]}')
    print(f'     Caminho : {raw_band_peaks_out_path.absolute()}')

    try:
        display(df_band_raw_peaks.head(3))
    except Exception:
        print(df_band_raw_peaks.head(3).to_string())


[OK] fingerprint_band_rank_raw_peaks.parquet salvo!
     Shape   : 260240 picos x 27 colunas
     Songs   : 1802
     Blocos  : 6506
     Caminho : C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\fingerprint_band_rank_raw_peaks.parquet


,song_id,filename,block_id,block_start_sec,block_end_sec,band_name,band_id,band_low_hz,band_high_hz,peak_rank_in_band,...,peak_time_relative_sec,peak_time_sec,midi_note,octave,semitone,pitch_class,peak_rank_global,valence,arousal,quadrant_label
0,2,2.mp3,0,15.0,25.0,low,0,20.0,250.0,1,...,8.289524,23.289524,47.785126,2.0,48.0,0.0,14,-0.08514,-0.137974,Q3_Low_Valence_Low_Arousal
1,2,2.mp3,0,15.0,25.0,low,0,20.0,250.0,2,...,6.594467,21.594467,44.628712,2.0,45.0,9.0,8,-0.08514,-0.137974,Q3_Low_Valence_Low_Arousal
2,2,2.mp3,0,15.0,25.0,low,0,20.0,250.0,3,...,2.670295,17.670296,50.453835,3.0,50.0,2.0,6,-0.08514,-0.137974,Q3_Low_Valence_Low_Arousal


In [12]:
# ============================================================
# VALIDACOES — fingerprint_band_rank_raw_peaks.parquet
# ============================================================

if df_band_raw_peaks.empty:
    print('[WARN] df_band_raw_peaks vazio — validacoes ignoradas.')
else:
    sr_val = SR_TARGET

    print('--- Shape ---')
    print(df_band_raw_peaks.shape)

    print('\n--- Primeiras linhas ---')
    try:
        display(df_band_raw_peaks.head(3))
    except Exception:
        print(df_band_raw_peaks.head(3).to_string())

    print('\n--- Colunas ---')
    print(df_band_raw_peaks.columns.tolist())

    print('\n--- Musicas e blocos ---')
    print(f'Musicas unicas : {df_band_raw_peaks["song_id"].nunique()}')
    print(f'Blocos unicos  : {df_band_raw_peaks[["song_id","block_id"]].drop_duplicates().shape[0]}')

    print('\n--- Picos por banda ---')
    if 'band_name' in df_band_raw_peaks.columns:
        try:
            display(df_band_raw_peaks.groupby('band_name', observed=True).size())
        except Exception:
            print(df_band_raw_peaks.groupby('band_name', observed=True).size().to_string())

    print('\n--- Picos por bloco (estatisticas) ---')
    try:
        display(df_band_raw_peaks.groupby(['song_id', 'block_id']).size().describe())
    except Exception:
        print(df_band_raw_peaks.groupby(['song_id', 'block_id']).size().describe().to_string())

    # --- Asserts obrigatorios ---
    required_cols = [
        'song_id', 'block_id', 'block_start_sec', 'block_end_sec',
        'frequency_hz', 'magnitude', 'magnitude_db', 'magnitude_norm',
        'peak_time_sec', 'peak_rank_global', 'peak_rank_in_band',
        'band_name', 'band_id', 'band_low_hz', 'band_high_hz', 'topk_per_band',
        'valence', 'arousal', 'quadrant_label',
    ]
    missing_cols = [c for c in required_cols if c not in df_band_raw_peaks.columns]
    assert not missing_cols, f'Colunas ausentes no raw_peaks: {missing_cols}'
    print('\n[OK] Todas as colunas obrigatorias presentes.')

    assert df_band_raw_peaks[['song_id', 'block_id', 'frequency_hz', 'magnitude']].notna().all().all(), \
        'Encontrados NaN em colunas criticas'

    assert df_band_raw_peaks['frequency_hz'].between(0, sr_val / 2).all(), \
        'Frequencias fora do range [0, Nyquist]'

    assert (df_band_raw_peaks['peak_time_sec'] >= df_band_raw_peaks['block_start_sec']).all(), \
        'Picos com peak_time_sec < block_start_sec'
    assert (df_band_raw_peaks['peak_time_sec'] < df_band_raw_peaks['block_end_sec']).all(), \
        'Picos com peak_time_sec >= block_end_sec'
    print('[OK] Todos os picos estao dentro dos limites temporais dos seus blocos.')

    # Sem sobreposicao entre blocos da mesma musica
    blocks_df = (
        df_band_raw_peaks[['song_id', 'block_id', 'block_start_sec', 'block_end_sec']]
        .drop_duplicates()
        .sort_values(['song_id', 'block_start_sec'])
    )
    for _sid, _g in blocks_df.groupby('song_id'):
        _g = _g.sort_values('block_start_sec').reset_index(drop=True)
        if len(_g) > 1:
            assert (
                _g['block_start_sec'].iloc[1:].to_numpy()
                >= _g['block_end_sec'].iloc[:-1].to_numpy()
            ).all(), f'Sobreposicao de blocos no raw_peaks para song_id={_sid}'
    print('[OK] Sem sobreposicao entre blocos no raw_peaks.')

    assert df_band_raw_peaks['topk_per_band'].max() == TOPK_PER_BAND, \
        f'TOPK_PER_BAND no raw_peaks ({df_band_raw_peaks["topk_per_band"].max()}) != {TOPK_PER_BAND}'
    print(f'[OK] TOPK_PER_BAND={TOPK_PER_BAND} preservado no raw_peaks.')

    print('\n[OK] Todas as validacoes passaram!')


--- Shape ---
(260240, 27)

--- Primeiras linhas ---


,song_id,filename,block_id,block_start_sec,block_end_sec,band_name,band_id,band_low_hz,band_high_hz,peak_rank_in_band,...,peak_time_relative_sec,peak_time_sec,midi_note,octave,semitone,pitch_class,peak_rank_global,valence,arousal,quadrant_label
0,2,2.mp3,0,15.0,25.0,low,0,20.0,250.0,1,...,8.289524,23.289524,47.785126,2.0,48.0,0.0,14,-0.08514,-0.137974,Q3_Low_Valence_Low_Arousal
1,2,2.mp3,0,15.0,25.0,low,0,20.0,250.0,2,...,6.594467,21.594467,44.628712,2.0,45.0,9.0,8,-0.08514,-0.137974,Q3_Low_Valence_Low_Arousal
2,2,2.mp3,0,15.0,25.0,low,0,20.0,250.0,3,...,2.670295,17.670296,50.453835,3.0,50.0,2.0,6,-0.08514,-0.137974,Q3_Low_Valence_Low_Arousal



--- Colunas ---
['song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec', 'band_name', 'band_id', 'band_low_hz', 'band_high_hz', 'peak_rank_in_band', 'topk_per_band', 'frame_idx', 'freq_bin', 'frequency_hz', 'magnitude', 'magnitude_db', 'magnitude_norm', 'peak_time_relative_sec', 'peak_time_sec', 'midi_note', 'octave', 'semitone', 'pitch_class', 'peak_rank_global', 'valence', 'arousal', 'quadrant_label']

--- Musicas e blocos ---
Musicas unicas : 1802
Blocos unicos  : 6506

--- Picos por banda ---


band_name
high         65060
low          65060
mid          65060
very_high    65060
dtype: int64


--- Picos por bloco (estatisticas) ---


count    6506.0
mean       40.0
std         0.0
min        40.0
25%        40.0
50%        40.0
75%        40.0
max        40.0
dtype: float64


[OK] Todas as colunas obrigatorias presentes.
[OK] Todos os picos estao dentro dos limites temporais dos seus blocos.
[OK] Sem sobreposicao entre blocos no raw_peaks.
[OK] TOPK_PER_BAND=10 preservado no raw_peaks.

[OK] Todas as validacoes passaram!


In [13]:
# ============================================================
# VALIDACOES
# ============================================================

_ok = 'df_band_rank' in dir() and df_band_rank is not None and not df_band_rank.empty

if not _ok:
    print('[WARN] df_band_rank nao disponivel para validacao.')
else:
    print('--- Shape ---')
    print(df_band_rank.shape)

    print('\n--- Primeiras linhas ---')
    try:
        display(df_band_rank.head(2))
    except NameError:
        print(df_band_rank.head(2).to_string())

    print('\n--- Colunas ---')
    print(df_band_rank.columns.tolist())

    print('\n--- Duracao dos blocos ---')
    print(df_band_rank[['block_duration_sec']].describe())

    print('\n--- Blocos menores que MIN_BLOCK_DURATION_SEC ---')
    curtos = df_band_rank[df_band_rank['block_duration_sec'] < MIN_BLOCK_DURATION_SEC]
    if curtos.empty:
        print('Nenhum bloco curto — OK')
    else:
        print(curtos[['song_id', 'block_id', 'block_duration_sec']])

    print('\n--- Blocos por musica ---')
    print(df_band_rank.groupby('song_id')['block_id'].count().describe())

    print('\n--- Nulos nas colunas principais ---')
    main_cols = [
        'song_id', 'block_id', 'block_start_sec', 'block_end_sec',
        'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label',
    ]
    print(df_band_rank[main_cols].isna().sum())

    print('\n--- Distribuicao dos quadrantes ---')
    print(df_band_rank['quadrant_label'].value_counts())

    print('\n--- Features por banda (cobertura de NaN) ---')
    for prefix in ['fp_low', 'fp_mid', 'fp_high', 'fp_very_high']:
        cols = [c for c in df_band_rank.columns if c.startswith(prefix)]
        nan_mean = df_band_rank[cols].isna().mean().mean() if cols else float('nan')
        print(f'  {prefix}: {len(cols)} colunas | NaN medio: {nan_mean:.4f}')

    print('\n--- Energia por banda ---')
    energy_cols = [
        'energy_db_low', 'energy_db_mid', 'energy_db_high', 'energy_db_very_high',
        'energy_norm_low', 'energy_norm_mid', 'energy_norm_high', 'energy_norm_very_high',
        'score_low', 'score_mid', 'score_high', 'score_very_high',
    ]
    avail = [c for c in energy_cols if c in df_band_rank.columns]
    try:
        display(df_band_rank[avail].describe())
    except NameError:
        print(df_band_rank[avail].describe().to_string())


--- Shape ---
(6506, 358)

--- Primeiras linhas ---


,song_id,filename,block_id,block_start_sec,block_end_sec,block_duration_sec,valence,arousal,quadrant,quadrant_label,...,fp_very_high_midi_mean,fp_very_high_midi_std,fp_very_high_unique_pitch_classes,fp_total_event_count,fp_total_mag_mean_db,fp_total_mag_std_db,fp_total_mag_mean_norm_block,fp_total_mag_std_norm_block,fp_total_freq_mean,fp_total_freq_std
0,2,2.mp3,0,15.0,25.0,10.0,-0.085140,-0.137974,Q3,Q3_Low_Valence_Low_Arousal,...,122.316243,0.858376,3,40,-38.353391,13.599803,0.043328,0.043862,3108.856201,3830.053182
1,2,2.mp3,1,25.0,35.0,10.0,-0.241535,-0.191833,Q3,Q3_Low_Valence_Low_Arousal,...,120.085102,0.861781,3,40,-41.513887,15.629449,0.051849,0.065483,3016.801758,3333.875349



--- Colunas ---
['song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec', 'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label', 'method', 'energy_db_low', 'energy_norm_low', 'energy_db_mid', 'energy_norm_mid', 'energy_db_high', 'energy_norm_high', 'energy_db_very_high', 'energy_norm_very_high', 'score_low', 'score_mid', 'score_high', 'score_very_high', 'fp_low_top_1_freq_hz', 'fp_low_top_1_magnitude_db', 'fp_low_top_1_magnitude_norm_block', 'fp_low_top_1_rank', 'fp_low_top_1_midi', 'fp_low_top_1_octave', 'fp_low_top_1_semitone_approx', 'fp_low_top_2_freq_hz', 'fp_low_top_2_magnitude_db', 'fp_low_top_2_magnitude_norm_block', 'fp_low_top_2_rank', 'fp_low_top_2_midi', 'fp_low_top_2_octave', 'fp_low_top_2_semitone_approx', 'fp_low_top_3_freq_hz', 'fp_low_top_3_magnitude_db', 'fp_low_top_3_magnitude_norm_block', 'fp_low_top_3_rank', 'fp_low_top_3_midi', 'fp_low_top_3_octave', 'fp_low_top_3_semitone_approx', 'fp_low_top_4_freq_hz', 'fp_low_top_4_magnitude_d

,energy_db_low,energy_db_mid,energy_db_high,energy_db_very_high,energy_norm_low,energy_norm_mid,energy_norm_high,energy_norm_very_high,score_low,score_mid,score_high,score_very_high
count,6506.000000,6506.000000,6506.000000,6506.000000,6506.000000,6506.000000,6506.000000,6.506000e+03,6506.000000,6506.000000,6506.000000,6.506000e+03
mean,-29.507632,-42.620310,-58.889442,-75.509986,0.103493,0.027513,0.005126,6.740976e-04,0.734179,0.224322,0.036718,4.780634e-03
std,9.086665,8.061348,10.463103,5.641095,0.050099,0.015359,0.004520,7.756337e-04,0.147246,0.135800,0.027273,5.517487e-03
min,-92.941727,-91.744431,-100.638771,-108.366905,0.000320,0.000838,0.000009,2.548525e-09,0.021646,0.006978,0.000099,1.712213e-08
25%,-33.630622,-47.295763,-65.965979,-79.255037,0.066765,0.016291,0.001990,1.519513e-04,0.676552,0.134233,0.017929,1.477478e-03
50%,-27.381639,-41.864128,-58.124228,-75.510082,0.102365,0.024993,0.003898,4.462747e-04,0.767874,0.189127,0.031245,3.462246e-03
75%,-23.013566,-36.841022,-50.886938,-71.649113,0.138259,0.035505,0.006944,9.083378e-04,0.831953,0.272617,0.048290,6.363200e-03
max,-14.527691,-22.834301,-33.418339,-57.372326,0.284609,0.108226,0.037131,7.711283e-03,0.992775,0.913288,0.337550,8.666551e-02


In [14]:
# ============================================================
# RESUMO FINAL
# ============================================================

_ok = 'df_band_rank' in dir() and df_band_rank is not None and not df_band_rank.empty

if not _ok:
    print('[WARN] df_band_rank nao disponivel para resumo.')
else:
    n_musicas = df_band_rank['song_id'].nunique()
    n_blocos  = len(df_band_rank)
    n_cols    = len(df_band_rank.columns)

    print('=' * 62)
    print('RESUMO FINAL — fingerprint_band_rank.parquet')
    print('=' * 62)
    print(f'  Musicas processadas              : {n_musicas}')
    print(f'  Total de blocos gerados          : {n_blocos}')
    print(f'  Blocos desc. duracao < {MIN_BLOCK_DURATION_SEC}s      : {stats["blocks_discarded_short"]}')
    print(f'  Blocos desc. ausencia de V/A     : {stats["blocks_discarded_no_va"]}')
    print(f'  Total de colunas                 : {n_cols}')
    print()
    print('  Distribuicao de quadrantes:')
    for q, cnt in df_band_rank['quadrant_label'].value_counts().items():
        print(f'    {q}: {cnt} ({100 * cnt / n_blocos:.1f}%)')
    print()
    print('  Colunas por banda:')
    for prefix in ['fp_low', 'fp_mid', 'fp_high', 'fp_very_high']:
        n = len([c for c in df_band_rank.columns if c.startswith(prefix)])
        print(f'    {prefix}: {n} colunas')
    print()
    print(f'  Arquivo salvo em: {FINGERPRINT_DIR / OUTPUT_PARQUET_NAME}')
    print(f'  Por musica em    : {FINGERPRINT_DIR / "songs"}')
    print('=' * 62)


RESUMO FINAL — fingerprint_band_rank.parquet
  Musicas processadas              : 1802
  Total de blocos gerados          : 6506
  Blocos desc. duracao < 5.0s      : 1757
  Blocos desc. ausencia de V/A     : 0
  Total de colunas                 : 358

  Distribuicao de quadrantes:
    Q1_High_Valence_High_Arousal: 3217 (49.4%)
    Q3_Low_Valence_Low_Arousal: 1382 (21.2%)
    Q2_Low_Valence_High_Arousal: 1019 (15.7%)
    Q4_High_Valence_Low_Arousal: 888 (13.6%)

  Colunas por banda:
    fp_low: 82 colunas
    fp_mid: 82 colunas
    fp_high: 82 colunas
    fp_very_high: 82 colunas

  Arquivo salvo em: C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\fingerprint_band_rank.parquet
  Por musica em    : C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\songs
